# DevOps in 15 Minutes: Ship Code Safely, Automatically, Often

**Audience:** beginners · **Length:** about 15–20 minutes · **Project:** `iris-mlops-pipeline`

| # | Section | Time |
|---|---------|------|
| 0 | Setup | 1 min |
| 1 | What is DevOps? | 2 min |
| 2 | Version control and versions | 3 min |
| 3 | Automated testing | 3 min |
| 4 | Containers with Docker | 3 min |
| 5 | Continuous Integration (CI) | 3 min |
| 6 | Operating the service | 2 min |
| 7 | Recap and Q&A | 2 min |

**How to use this notebook:** run the cells from top to bottom. Along the way you'll find four kinds of notes:

- 💡 **Why this matters**: the key idea behind each step
- 🤔 **Think about it**: questions to reflect on or discuss
- 🛠️ **Try it yourself**: small hands-on exercises
- 📌 **Tip**: practical advice for running the notebook smoothly

> 📌 **Tip:** Docker builds take a minute or two the first time. Run section 4 once in advance, and later rebuilds will take only seconds thanks to Docker's cache. If Docker isn't installed (for example in Google Colab), the notebook automatically shows sample output instead, so the demo still works.

In [9]:
# ---------------------------------------------------------------
# SETUP — run this cell first. It works locally AND in Google Colab.
# Locally, choose the project's ".venv" kernel (created by `uv sync`).
# ---------------------------------------------------------------
import os          # change folders
import sys         # check where we are running and which Python
import subprocess  # run terminal commands from Python
from pathlib import Path

# 👉 Replace ambi1999 with your GitHub username (only needed for Colab).
GITHUB_REPO = "https://github.com/ambi1999/iris-mlops-pipeline.git"

IN_COLAB = "google.colab" in sys.modules  # True when running in Google Colab

if IN_COLAB:
    # Colab starts with an empty folder, so download the project from GitHub.
    if Path.cwd().name != "iris-mlops-pipeline":  # skip if this cell was already run
        if not Path("iris-mlops-pipeline").exists():
            subprocess.run(["git", "clone", GITHUB_REPO], check=True)
        os.chdir("iris-mlops-pipeline")
    # Colab has its own Python, so we use uv to turn uv.lock into a plain
    # package list and install those exact versions into Colab's Python.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "export", "--frozen", "--all-groups",
                    "--no-hashes", "--no-emit-project", "-o", "colab-requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab-requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    # Locally, the notebook starts inside notebooks/. Step up to the project
    # root so that train.py, app.py, etc. can be found.
    os.chdir("..")


def run(command):
    """Run a terminal command and print its output, like typing it in a terminal."""
    print(f"$ {' '.join(command)}\n")
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout + result.stderr)
    print(f"(exit code {result.returncode}: {'success ✅' if result.returncode == 0 else 'failure ❌'})")
    return result


print("Running in:", "Google Colab" if IN_COLAB else "local Jupyter")
print("Project folder:", Path.cwd())

Running in: local Jupyter
Project folder: c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline


## 1. What is DevOps? *(≈2 min)*

Traditionally, **developers** wrote code and threw it "over the wall" to **operations**, who had to run it. Releases were rare, big, scary, and often broke things.

**DevOps** combines the two: the same team builds *and* runs the software, with heavy **automation** in between.

```
   Plan → Code → Build → Test → Release → Deploy → Operate → Monitor
    ▲                                                          │
    └──────────────── feedback, then repeat ◄──────────────────┘
```

Core ideas:
- **Automate everything repeatable:** testing, building, deploying.
- **Small, frequent changes:** easier to test and easier to undo.
- **Fast feedback:** find out in minutes, not weeks, that something broke.

> 💡 **Why this matters:** Think of a car factory. One approach is to build each car by hand and hope it starts at the end. The other is an assembly line with an inspection station after every step. DevOps is the assembly line: problems are caught early, close to where they were introduced, when they're cheap to fix.

> 🤔 **Think about it:** Have you ever heard, or said, "but it works on my machine"? That phrase is the problem DevOps exists to solve.

## 2. Version control and versions *(≈3 min)*

Everything starts with **Git**: a full history of every change, who made it, and why. GitHub stores it online and is where our automation runs.

An important habit: commit the **recipe**, not the things the recipe produces. The `.gitignore` file lists what Git should skip.

In [10]:
# Show which files Git is told to ignore (and why, in the comments).
print(Path(".gitignore").read_text())

# If this folder is a Git repository, show the latest commits.
if Path(".git").exists():
    run(["git", "log", "--oneline", "-5"])
else:
    print("(Not a Git repository yet. After you push to GitHub, this shows your commit history.)")

# ------------------------------------------------------------------
# .gitignore â€” files Git should NOT track or upload to GitHub.
# Rule of thumb: commit the RECIPE (code), not things the recipe makes.
# Note: comments must be on their own line in this file.
#
# DO commit: pyproject.toml, uv.lock and .python-version. Together they
# let anyone recreate the exact environment with `uv sync`.
# ------------------------------------------------------------------

# Compiled Python files, created automatically
__pycache__/

# Local virtual environment: recreate it with `uv sync`
.venv/

# Trained model and its metrics: recreate them with `uv run python train.py`
*.joblib
metrics.json

# pytest's local cache
.pytest_cache/

# Jupyter's autosave copies of notebooks
.ipynb_checkpoints/

# Temporary package list created by the notebooks when run in Google Colab
colab-requirements.txt

(Not a Git repository yet. After you push to GitHub, this shows your commit history.)


> 💡 **Why this matters:** `*.joblib` is ignored because the trained model doesn't need to be committed: anyone can recreate it by running `train.py`. The same goes for `.venv/`, since `uv sync` recreates it from `uv.lock`. This keeps the repository small and makes the *code* the single source of truth.

### One source of truth for versions

"Works on my machine" often comes down to *versions*: a different Python, or a different numpy. This project fixes both, and three files work together:

| File | What it says | Who writes it |
|------|--------------|---------------|
| `.python-version` | Use Python 3.13 | You (or `uv python pin 3.13`) |
| `pyproject.toml` | The packages we need, with minimum versions | You (or `uv add <package>`) |
| `uv.lock` | The **exact** version of every package, including hidden ones | uv, automatically |

Laptop, CI and Docker all read these same files, so they can't drift apart.

In [11]:
import tomllib  # built into Python: reads .toml files

print(".python-version says:", Path(".python-version").read_text().strip())

# Packages WE asked for (pyproject.toml)...
project = tomllib.loads(Path("pyproject.toml").read_text())
direct = project["project"]["dependencies"] + project["dependency-groups"]["dev"]
print(f"\npyproject.toml lists {len(direct)} packages we chose ourselves.")

# ...versus everything uv actually locked (uv.lock).
if Path("uv.lock").exists():
    locked = tomllib.loads(Path("uv.lock").read_text())["package"]
    print(f"uv.lock pins {len(locked)} packages exactly. A few of them:")
    for pkg in locked[:8]:
        print(f"  {pkg['name']:<20} {pkg.get('version', '')}")
else:
    print("\nuv.lock not found. Run `uv sync` in the terminal to create it.")

.python-version says: 3.13

pyproject.toml lists 8 packages we chose ourselves.
uv.lock pins 121 packages exactly. A few of them:
  annotated-doc        0.0.6
  annotated-types      0.8.0
  anyio                4.16.0
  appnope              1.0.0
  argon2-cffi          25.1.0
  argon2-cffi-bindings 26.1.0
  arrow                1.4.0
  asttokens            3.0.2


> 💡 **Why this matters:** Look at the gap between the two numbers. We chose about 8 packages, but they quietly pull in many more. Without a lock file, any of those hidden packages could update overnight and change your results. With `uv.lock` committed, everyone gets identical versions. To keep them current safely, **Dependabot** (`.github/dependabot.yml`) opens a pull request with updated versions once a week, and CI tests it before anyone merges. Upgrades become small, safe and boring, which is exactly what you want.

## 3. Automated testing *(≈3 min)*

Tests are small programs that check your program. Our project has three, in `test_app.py`:
1. The health endpoint answers.
2. A known flower gets the right species.
3. Bad input is rejected instead of crashing.

The model file must exist before testing, so we train first.

In [12]:
run([sys.executable, "train.py"])            # create model.joblib (needed by the app)
run([sys.executable, "-m", "pytest", "-v"])  # run every test in the project

$ c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline\.venv\Scripts\python.exe train.py

Test accuracy: 1.000
Model saved to model.joblib

(exit code 0: success ✅)
$ c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline\.venv\Scripts\python.exe -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline
configfile: pyproject.toml
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 3 items

test_app.py::test_health PASSED                                          [ 33%]
test_app.py::test_predict_setosa PASSED                                  [ 66%]
test_app.py::test_bad_input_rejected PASSED                              [100%]

============================== warnings summary ===============================
.venv\Li

CompletedProcess(args=['c:\\Ambi\\workspace\\01MLOpsProject\\iris-mlops-pipeline\\.venv\\Scripts\\python.exe', '-m', 'pytest', '-v'], returncode=0, stdout='\x1b============================= test session starts =============================\x1b\nplatform win32 -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- c:\\Ambi\\workspace\\01MLOpsProject\\iris-mlops-pipeline\\.venv\\Scripts\\python.exe\ncachedir: .pytest_cache\nrootdir: c:\\Ambi\\workspace\\01MLOpsProject\\iris-mlops-pipeline\nconfigfile: pyproject.toml\nplugins: anyio-4.16.0, platformdirs-4.14.5\n\x1bcollecting ... \x1bcollected 3 items\n\ntest_app.py::test_health \x1bPASSED\x1b\x1b                                          [ 33%]\x1b\ntest_app.py::test_predict_setosa \x1bPASSED\x1b\x1b                                  [ 66%]\x1b\ntest_app.py::test_bad_input_rejected \x1bPASSED\x1b\x1b                              [100%]\x1b\n\n\x1b============================== warnings summary ===============================\x1b\n.venv\\Lib\\sit

### Now let's break something on purpose

Imagine a teammate changes something and the app now returns the wrong answer. We'll add a temporary test that expects the wrong species and see what happens.

In [13]:
# Write a temporary test with a deliberately wrong expectation.
broken_test = Path("test_broken_demo.py")
broken_test.write_text(
    "from fastapi.testclient import TestClient\n"
    "from app import app\n\n"
    "def test_wrong_expectation():\n"
    "    flower = {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}\n"
    "    response = TestClient(app).post('/predict', json=flower)\n"
    "    assert response.json()['species'] == 'virginica'  # wrong on purpose!\n"
)

try:
    run([sys.executable, "-m", "pytest", "-q", "test_broken_demo.py"])
finally:
    broken_test.unlink()  # always clean up the temporary test file
    print("\n(Temporary test file removed.)")

$ c:\Ambi\workspace\01MLOpsProject\iris-mlops-pipeline\.venv\Scripts\python.exe -m pytest -q test_broken_demo.py

F                                                                        [100%]
================================== FAILURES ===================================
___________________________ test_wrong_expectation ____________________________

    def test_wrong_expectation():
        flower = {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}
        response = TestClient(app).post('/predict', json=flower)
>       assert response.json()['species'] == 'virginica'  # wrong on purpose!
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       AssertionError: assert 'setosa' == 'virginica'
E         
E         - virginica
E         + setosa

test_broken_demo.py:7: AssertionError
============================== warnings summary ===============================
.venv\Lib\site-packages\fastapi\testclient.py:1
  c:\Ambi\workspace\01MLOpsProject\ir

> 💡 **Why this matters:** The failure message shows exactly what was expected versus what came back, and the computer caught the problem in seconds, before any user saw it. In CI, a red result like this blocks the change from being merged. Tests also let people change code with confidence: if they break something, they'll know immediately.

## 4. Containers with Docker *(≈4 min)*

"It works on my machine" happens because machines differ: Python versions, installed packages, operating systems. A **Docker container** packages the app *together with* everything it needs, so it runs identically everywhere.

```
 Dockerfile  ──docker build──►  Image  ──docker run──►  Container
 (recipe)                      (frozen package)         (running app)
```

> 💡 **Why this matters:** Think of shipping containers. Before they were standardised, every ship loaded cargo differently. Afterwards, any crane, ship or truck could handle any container. Docker does the same for software: any server that runs Docker can run our app.

In [14]:
# Show the recipe. Every line is commented in the file itself.
print(Path("Dockerfile").read_text())

# ------------------------------------------------------------------
# Dockerfile â€” Step 4 of the pipeline: PACKAGE everything into an image.
#
# Build the image:   docker build -t iris-mlops-pipeline .
# Run a container:   docker run -p 8000:8000 iris-mlops-pipeline
# Then open:         http://localhost:8000/docs
#
# Docker reads this file top to bottom. Each instruction creates a
# "layer"; Docker caches layers and only rebuilds what changed.
# ------------------------------------------------------------------

# Build arguments: values that can be changed at build time with
#   docker build --build-arg PYTHON_VERSION=3.13 ...
# CI passes the value from .python-version, so there is ONE source of truth.
# The default below must match .python-version for local builds.
ARG PYTHON_VERSION=3.13
# Tip: replace "latest" with your local uv version (run: uv --version),
# e.g. 0.9.5, so the image always uses exactly the same uv.
ARG UV_VERSION=latest

# Stage 1: grab the uv program from its 

In [15]:
import shutil, time, json, urllib.request

IMAGE = "iris-mlops-pipeline"

def docker_available():
    """True if Docker is installed AND its engine is running."""
    if shutil.which("docker") is None:
        return False
    return subprocess.run(["docker", "info"], capture_output=True).returncode == 0

if docker_available():
    print("Docker found, running live.\n")
    python_version = Path(".python-version").read_text().strip()   # single source of truth
    run(["docker", "build", "--build-arg", f"PYTHON_VERSION={python_version}",
         "-t", IMAGE, "."])                                       # 1. build the image
    run(["docker", "run", "-d", "--rm", "-p", "8000:8000",
         "--name", "iris-demo", IMAGE])                          # 2. start a container in the background
    try:
        # 3. Wait until the app inside the container answers its health check.
        for _ in range(30):
            try:
                with urllib.request.urlopen("http://localhost:8000/health") as r:
                    print("Health check:", r.read().decode())
                    break
            except Exception:
                time.sleep(1)
        # 4. Ask the containerised model for a prediction over real HTTP.
        body = json.dumps({"sepal_length": 6.7, "sepal_width": 3.0,
                           "petal_length": 5.2, "petal_width": 2.3}).encode()
        req = urllib.request.Request("http://localhost:8000/predict", data=body,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req) as r:
            print("Prediction from container:", r.read().decode())
    finally:
        run(["docker", "stop", "iris-demo"])                     # 5. always clean up
else:
    print("Docker isn't available here, so here is sample output from a real run:\n")
    print("""$ docker build --build-arg PYTHON_VERSION=3.13 -t iris-mlops-pipeline .
[+] Building 23.8s (12/12) FINISHED
 => [stage-1 1/7] FROM docker.io/library/python:3.13-slim
 => [stage-1 2/7] COPY --from=uv /uv /uvx /bin/
 => [stage-1 3/7] WORKDIR /app
 => [stage-1 4/7] COPY pyproject.toml uv.lock .python-version ./
 => [stage-1 5/7] RUN uv sync --locked --no-dev --no-install-project
 => [stage-1 6/7] COPY train.py app.py ./
 => [stage-1 7/7] RUN python train.py      (Test accuracy: 1.000)
 => naming to docker.io/library/iris-mlops-pipeline

$ docker run -d --rm -p 8000:8000 --name iris-demo iris-mlops-pipeline
3f9c2a71be04...

$ curl http://localhost:8000/health
{"status":"ok"}

$ curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \\
       -d '{"sepal_length":6.7,"sepal_width":3.0,"petal_length":5.2,"petal_width":2.3}'
{"species":"virginica"}

$ docker stop iris-demo""")

Docker isn't available here, so here is sample output from a real run:

$ docker build --build-arg PYTHON_VERSION=3.13 -t iris-mlops-pipeline .
[+] Building 23.8s (12/12) FINISHED
 => [stage-1 1/7] FROM docker.io/library/python:3.13-slim
 => [stage-1 2/7] COPY --from=uv /uv /uvx /bin/
 => [stage-1 3/7] WORKDIR /app
 => [stage-1 4/7] COPY pyproject.toml uv.lock .python-version ./
 => [stage-1 5/7] RUN uv sync --locked --no-dev --no-install-project
 => [stage-1 6/7] COPY train.py app.py ./
 => [stage-1 7/7] RUN python train.py      (Test accuracy: 1.000)
 => naming to docker.io/library/iris-mlops-pipeline

$ docker run -d --rm -p 8000:8000 --name iris-demo iris-mlops-pipeline
3f9c2a71be04...

$ curl http://localhost:8000/health
{"status":"ok"}

$ curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
       -d '{"sepal_length":6.7,"sepal_width":3.0,"petal_length":5.2,"petal_width":2.3}'
{"species":"virginica"}

$ docker stop iris-demo


> 💡 **Why this matters:** Follow the five numbered steps in the code comments. The container came with its *own* Python, its *own* packages and its *own* trained model; the setup of the computer running it didn't matter. Training happens *inside* the build (`RUN python train.py`), so if the model fails its quality gate, no image is produced at all. Rebuilds are fast thanks to **layer caching**: Docker only redoes steps whose inputs changed.

## 5. Continuous Integration (CI) *(≈3 min)*

Everything we just did by hand (install, train, test, build), GitHub does **automatically on every push**, on a fresh machine. That's **Continuous Integration**. The instructions live in `.github/workflows/ci.yml`.

```
 git push ──► GitHub Actions starts a fresh Ubuntu machine
              ├─ 1. Check out the code
              ├─ 2. Install uv
              ├─ 3. uv sync: Python + exact packages from uv.lock
              ├─ 4. Train model        ← quality gate (MLOps)
              ├─ 5. Run tests          ← catches bugs
              ├─ 6. Upload model artifact
              └─ 7. Build Docker image ← proves it packages
              Any step fails → ❌ red run, change is blocked
              All steps pass → ✅ green run
```

In [16]:
# Show the pipeline definition. Every line is commented in the file itself.
print(Path(".github/workflows/ci.yml").read_text())

UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 438: character maps to <undefined>

> 🛠️ **Try it yourself:** Open your repository's **Actions** tab on GitHub to see a real run with its green checkmarks. Then set `MIN_ACCURACY = 1.01` in `train.py`, commit and push, and watch the pipeline go red at the training step. Change it back, push again, and it turns green.

> 📌 **Tip:** Keep an eye on the **CI badge** at the top of the README. It shows anyone visiting the repository, at a glance, whether the project currently works.

> 💡 **Why this matters:** The natural next step after CI is **CD (Continuous Delivery/Deployment)**: when the pipeline is green, automatically push the Docker image to a registry and deploy it to a server.

## 6. Operating the service *(≈2 min)*

DevOps doesn't stop at deployment; "you build it, you run it." Two simple operational signals:
- **Health checks:** load balancers and tools like Kubernetes call `/health` regularly, and restart the app if it stops answering.
- **Latency:** how fast are we responding? Slow is often the first sign of trouble.

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)
print("Health check:", client.get("/health").json())

# Measure how long 200 predictions take.
flower = {"sepal_length": 5.9, "sepal_width": 3.0, "petal_length": 4.2, "petal_width": 1.5}
start = time.perf_counter()
for _ in range(200):
    client.post("/predict", json=flower)
elapsed = time.perf_counter() - start
print(f"200 predictions in {elapsed:.2f}s → about {elapsed / 200 * 1000:.1f} ms each")

> 💡 **Why this matters:** In production, numbers like these go to dashboards and alerting tools (for example Prometheus and Grafana), so the team hears about problems *before* users complain. Monitoring feeds back into planning, which closes the DevOps loop from section 1.

## 7. Recap *(≈2 min)*

| DevOps idea | Where it lives in this repo |
|-------------|-----------------------------|
| Version control | Git + GitHub, `.gitignore` |
| Automated testing | `test_app.py` (pytest) |
| Reproducible environments | `.python-version`, `pyproject.toml`, `uv.lock`, `Dockerfile` |
| Automated updates | `.github/dependabot.yml` |
| Containers | `Dockerfile`, `.dockerignore` |
| Continuous Integration | `.github/workflows/ci.yml` |
| Operations | `/health` endpoint, latency check |

**DevOps vs MLOps in one sentence:** DevOps automates shipping *code*; MLOps applies the same ideas to shipping *code + data + models*, and adds quality gates and drift monitoring for the model.

🤔 **Think about it:**
- What's one manual step in your work that a pipeline could automate?
- What would be the first test you'd write for your own project?

> 💡 **Why this matters:** Everything in this notebook is free to use: GitHub Actions, Docker, pytest and uv. You can fork this repository and have your own working pipeline in about ten minutes.